# Анализ табличных данных: Medical Insurance Price Prediction

**Автор:** Матвиевский Дмитрий Денисович  
**Группа:** ЕТ-142  
**Кейс:** №58 – Прогнозирование стоимости медицинской страховки  

## Постановка задачи

- **Тип задачи:** Регрессия (предсказание непрерывного значения)
- **Входные данные:** демографические и медицинские показатели клиента (возраст, пол, ИМТ, количество детей, статус курения, регион)
- **Выходные данные:** годовая стоимость медицинской страховки (charges)
- **Предметная область:** медицинское страхование, финансы
- **Практическая значимость:** помощь страховым компаниям в обоснованном ценообразовании, выявление факторов, влияющих на стоимость полиса

In [ ]:
# Установка и импорт необходимых библиотек
!pip install -q pandas numpy matplotlib seaborn scikit-learn plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats

# Настройки визуализации
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
pd.set_option('display.max_columns', None)

print("Библиотеки успешно загружены")

In [ ]:
# Загрузка данных
url = 'https://raw.githubusercontent.com/XTR1DE/coursework-data-analysis/main/data/tabular/Medical_insurance.csv'
df = pd.read_csv(url)

print("=== 1. ОБЩЕЕ ОПИСАНИЕ ДАТАСЕТА ===\n")

# 1.1 Название, ссылка, лицензия, мощность
print(f"Название: Medical Insurance Price Prediction")
print(f"Источник: Kaggle (https://www.kaggle.com/datasets/harishkumardatalab/medical-insurance-price-prediction)")
print(f"Лицензия: CC0 (общественное достояние)")
print(f"Количество записей: {df.shape[0]}")
print(f"Количество признаков: {df.shape[1]}\n")

# 1.2 Перечень признаков
print("=== ПРИЗНАКИ === (тип, семантика, пример значения)")
print("age         : int     – возраст клиента (18–64)")
print("sex         : object  – пол (female, male)")
print("bmi         : float   – индекс массы тела (реальный)")
print("children    : int     – количество детей (0–5)")
print("smoker      : object  – курит? (yes, no)")
print("region      : object  – регион (southwest, southeast, northwest, northeast)")
print("charges     : float   – годовая стоимость страховки (целевая)\n")

# 1.3 Пропуски
missing = df.isnull().sum()
print("=== ПРОПУСКИ ===")
if missing.sum() == 0:
    print("Пропущенные значения отсутствуют во всех признаках.")
else:
    print(missing[missing > 0])

# 1.4 Дубликаты
duplicates = df.duplicated().sum()
print(f"\n=== ДУБЛИКАТЫ ===")
print(f"Количество дублированных строк: {duplicates}")

# 1.5 Чувствительность данных
print("\n=== ЧУВСТВИТЕЛЬНОСТЬ ===")
print("Датасет не содержит персонально идентифицируемой информации (анонимные демографические данные).")
print("Возможна чувствительность: признак 'smoker' (медицинская информация), но без привязки к личности.")

## 1.2 ХАРАКТЕРИСТИКА ЧИСЛОВЫХ ДАННЫХ

In [ ]:

# 1.2.1 Диаграмма распределения
num_features = ['age', 'bmi', 'children', 'charges']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].hist(df[col], bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Распределение признака {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Частота')

plt.tight_layout()
plt.show()

# Краткий вывод
print("• age: равномерное распределение от 18 до 64")
print("• bmi: близко к нормальному, пик в районе 30")
print("• children: дискретное, большинство 0–2")
print("• charges: сильная асимметрия – редкие очень высокие счета")

In [ ]:
# 1.2.2 Визуализация пар признаков
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Пара 1: age и charges
sns.scatterplot(data=df, x='age', y='charges', hue='smoker', alpha=0.6, ax=axes[0])
axes[0].set_title('Стоимость и возраст (цвет – курит)')

# Пара 2: bmi и charges
sns.scatterplot(data=df, x='bmi', y='charges', hue='smoker', alpha=0.6, ax=axes[1])
axes[1].set_title('Стоимость и ИМТ (цвет – курит)')

# Пара 3: children и charges
mean_charges = df.groupby('children')['charges'].mean()
axes[2].bar(mean_charges.index, mean_charges.values, color='skyblue', edgecolor='black')
axes[2].set_title('Средняя стоимость по количеству детей')
axes[2].set_xlabel('children')
axes[2].set_ylabel('charges (среднее)')

plt.tight_layout()
plt.show()

print("• Курящие платят намного больше.")
print("• С возрастом и ИМТ стоимость растёт, но у некурящих рост умеренный.")
print("• Количество детей почти не влияет на среднюю стоимость.")

In [ ]:
# 1.2.3 Визуализация с plotly
# График 1:
fig1 = px.scatter(df, x='age', y='charges', color='smoker',
                  title='Стоимость страховки и возраст',
                  labels={'charges':'Стоимость', 'age':'Возраст'},
                  hover_data=['bmi', 'children'])
fig1.show()

# График 2: средний ИМТ по регионам
mean_bmi = df.groupby('region')['bmi'].mean()
plt.figure(figsize=(8, 5))
plt.bar(mean_bmi.index, mean_bmi.values, color='mediumseagreen', edgecolor='black')
plt.title('Средний индекс массы тела по регионам')
plt.xlabel('Регион')
plt.ylabel('BMI')
plt.show()

In [ ]:
# 1.2.5 Тепловые карты

# Карта 1: средняя стоимость страховки по комбинации регион + курильщик
pivot_table = df.pivot_table(index='region', columns='smoker', values='charges', aggfunc='mean')
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_table, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5, cbar_kws={'label': 'Средняя charges'})
plt.title('Тепловая карта: средняя стоимость страховки (регион и курит)')
plt.xlabel('Курит')
plt.ylabel('Регион')
plt.show()

# Карта 2: тепловая карта пропусков
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='binary',
            vmin=0, vmax=1, mask=df.isnull()==False)

from matplotlib.colors import ListedColormap
cmap = ListedColormap(['white', 'black'])
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap=cmap)
plt.title('Тепловая карта пропусков (белый – нет пропуска, чёрный – пропуск)')
plt.show()

In [ ]:
# 1.2.6 Удаление дубликатов
dups_before = df.duplicated().sum()
df = df.drop_duplicates()
dups_after = df.duplicated().sum()
print(f"Дубликатов до удаления: {dups_before}")
print(f"Дубликатов после удаления: {dups_after}")
print(f"Удалено строк: {dups_before - dups_after}")

In [ ]:
# 1.2.7 Выявление выбросов для числовых признаков
outliers_summary = {}
for col in num_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    outliers_summary[col] = len(outliers)
    print(f"{col}: {len(outliers)} выбросов ({(len(outliers)/len(df))*100:.1f}%)")

# Boxplot для визуализации
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, col in enumerate(num_features):
    sns.boxplot(y=df[col], ax=axes[i])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

print("• Наибольшее число выбросов – в признаке 'charges'. Это соответствует высокой стоимости у курящих.")
print("• В 'bmi' тоже есть выбросы (экстремальный вес).")
print("• Для регрессии выбросы можно не удалять, т.к. они отражают реальные случаи.")

In [ ]:
# 1.2.8 Условная фильтрация сэмплов

# Фильтр 1: только курящие
smoker_df = df[df['smoker'] == 'yes']
print(f"1. Курящие клиенты: {len(smoker_df)}")

# Фильтр 2: с детьми > 2
many_children = df[df['children'] > 2]
print(f"2. с детьми > 2: {len(many_children)}")

# Фильтр 3: курящие старше 50 лет с ИМТ > 30
high_risk = df[(df['smoker'] == 'yes') & (df['age'] > 50) & (df['bmi'] > 30)]
print(f"3. Курящие, возраст > 50, ИМТ > 30: {len(high_risk)}")

# Визуализация средней стоимости по группам
filtered_means = {
    'Все': df['charges'].mean(),
    'Курящие': smoker_df['charges'].mean(),
    'Детей>2': many_children['charges'].mean(),
    'Курящие+возраст>50+ИМТ>30': high_risk['charges'].mean()
}
plt.figure(figsize=(10, 5))
plt.bar(filtered_means.keys(), filtered_means.values(), color='skyblue')
plt.title('Средняя стоимость страховки в разных группах')
plt.ylabel('Средние charges')
plt.show()

print("• Курящие платят в среднем в 3–4 раза больше некурящих.")
print("• Наличие более 2 детей не сильно повышает среднюю стоимость.")
print("• Группа 'курящие + возраст > 50 + ИМТ > 30' имеет максимальную среднюю стоимость (даже выше, чем просто курящие).")

In [ ]:
# 1.2.9 Добавление шума в признаки age и bmi
np.random.seed(42)
noise_age = np.random.normal(0, 2, size=len(df))
noise_bmi = np.random.normal(0, 1.5, size=len(df))

df_noisy = df.copy()
df_noisy['age_noisy'] = df['age'] + noise_age
df_noisy['bmi_noisy'] = df['bmi'] + noise_bmi

# Визуализация исходных и зашумлённых распределений
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['age'], bins=30, alpha=0.5, label='Исходный age')
axes[0].hist(df_noisy['age_noisy'], bins=30, alpha=0.5, label='Age + шум')
axes[0].legend()
axes[0].set_title('Шум в age')

axes[1].hist(df['bmi'], bins=30, alpha=0.5, label='Исходный bmi')
axes[1].hist(df_noisy['bmi_noisy'], bins=30, alpha=0.5, label='BMI + шум')
axes[1].legend()
axes[1].set_title('Шум в bmi')
plt.tight_layout()
plt.show()

In [ ]:
# 1.2.10 Преобразование age в возрастные категории
bins = [18, 30, 40, 50, 65]
labels = ['18-29', '30-39', '40-49', '50-64']
df['Групповой возраст'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)

print("Распределение возрастных групп:")
print(df['Групповой возраст'].value_counts().sort_index())

# Сравнение средней стоимости по группам
group_charges = df.groupby('Групповой возраст', observed=False)['charges'].mean()
group_charges.plot(kind='bar', color='teal')
plt.title('Средняя стоимость страховки по возрастным группам')
plt.ylabel('charges')
plt.show()

In [ ]:
# 1.2.12 Оценка изменений после фильтрации
df_original = pd.read_csv(url)
dups_before = df_original.duplicated().sum()
print(f"Дубликатов в исходном датасете: {dups_before}")

dups_after = df.duplicated().sum()
print(f"Дубликатов после удаления: {dups_after}")

plt.figure(figsize=(10, 5))
plt.hist(df_original['charges'], bins=50, alpha=0.6, label='До удаления дубликатов', color='blue')
plt.hist(df['charges'], bins=50, alpha=0.6, label='После удаления дубликатов', color='orange')
plt.title('Сравнение распределения charges до и после удаления дубликатов')
plt.xlabel('charges')
plt.ylabel('Частота')
plt.legend()
plt.show()

## 3. ХАРАКТЕРИСТИКА КАТЕГОРИАЛЬНЫХ ДАННЫХ

In [ ]:
# 1.3.1 Перечень категорий для каждого категориального признака
cat_features = ['sex', 'smoker', 'region']
for col in cat_features:
    print(f"{col}: {df[col].unique().tolist()}")

In [ ]:
# 1.3.2 Диаграммы распределения категориальных данных
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(cat_features):
    df[col].value_counts().plot(kind='bar', ax=axes[i], color=['coral', 'skyblue'])
    axes[i].set_title(f'Распределение {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Количество')
plt.tight_layout()
plt.show()

In [ ]:
# 1.3.3 Преобразование категориальных в числовые (Label Encoding и One-Hot)
from sklearn.preprocessing import LabelEncoder

# Label Encoding для smoker и sex (бинарные)
le_sex = LabelEncoder()
le_smoker = LabelEncoder()
df['sex_enc'] = le_sex.fit_transform(df['sex'])
df['smoker_enc'] = le_smoker.fit_transform(df['smoker'])

# One-Hot Encoding для region
df_region_encoded = pd.get_dummies(df['region'], prefix='region')
df = pd.concat([df, df_region_encoded], axis=1)

print("Пример после кодирования:")
display(df[['sex', 'sex_enc', 'smoker', 'smoker_enc', 'region', 'region_northeast', 'region_northwest', 'region_southeast', 'region_southwest']].head())

In [ ]:
# 1.3.5 Введение новой категории: группа риска на основе курения и высокого ИМТ
def risk_group(row):
    if row['smoker'] == 'yes' and row['bmi'] > 30:
        return 'high_risk'
    elif row['smoker'] == 'yes' or row['bmi'] > 30:
        return 'medium_risk'
    else:
        return 'low_risk'

df['risk_category'] = df.apply(risk_group, axis=1)
print("Распределение новой категории 'risk_category':")
print(df['risk_category'].value_counts())

# Визуализация средней стоимости по группам риска
sns.barplot(data=df, x='risk_category', y='charges', order=['low_risk', 'medium_risk', 'high_risk'])
plt.title('Средняя стоимость страховки по категориям риска')
plt.show()

print("=== ВЫВОД (3.5) ===")
print("• Новая категория 'risk_category' объединяет влияние курения и ожирения.")
print("• Высокий риск даёт максимальную стоимость, низкий – минимальную.")

4. ОСОБЕННОСТИ, ВОЗМОЖНОСТИ И ПОТЕНЦИАЛЬНЫЕ РИСКИ ИСПОЛЬЗОВАНИЯ ДАТАСЕТА

In [ ]:
# 1.4.1 Особенности предметной области
print("• Датасет создан для демонстрации влияния демографических факторов на стоимость медстраховки")
print("• Главная особенность – фактор 'курит/не курит', который перекрывает влияние других признаков")
print("• В реальности страховые компании используют больше признаков (история болезней, семейный анамнез и др)")


## 5. Гиппотезы

In [ ]:
# 1.4.2 Гипотезы для корректного использования
print("1. Гипотеза: Регион влияет на стоимость не напрямую, а через доступность медицинских услуг.")
print("   => При переносе модели на другие страны потребуется переобучение.")
print("2. Гипотеза: Выбросы в charges – это реальные клиенты с тяжелыми заболеваниями, удалять их нельзя.")

In [ ]:
# 1.4.3 Гипотезы для неэтичного использования
print("1. Дискриминация по возрасту: модель может необоснованно завышать стоимость для пожилых,")
print("   даже если они здоровы. Это может привести к возрастной дискриминации.")
print("2. Дискриминация по региону: если в каком-то регионе исторически выше цены,")
print("   модель может ущемлять жителей этого региона без учёта индивидуальных факторов.")

Потенциальные возможности расширения датасета.

In [ ]:
print("【Признаки для добавления】")
print("1. Медицинская история (хронические заболевания, давление, холестерин)")
print("2. Количество страховых случаев за прошлые годы")
print("3. Данные о физической активности (тренировки)")
print("4. Добавить временную компоненту (изменение стоимости по годам) – для прогнозирования трендов.")

print("\n【Новые задачи, которые можно решать】")
print("1. Кластеризация клиентов для разработки персонализированных тарифов")
print("2. Прогнозирование вероятности наступления страхового случая (бинарная классификация)")
print("3. Рекомендательная система дополнительных услуг (стоматология, спорт)")